# Explore LightHit results

This notebook reads the same exported data as the results page. It does not run transport or select acceptance thresholds. Historical arrays retain their recorded source and date.


In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

data = json.loads(Path("results.json").read_text())
case_id = "electron-100GeV"
case = next(x for x in data["examples"] if x["id"] == case_id)
print(case["title"], case["date"], case["source"])
print("Response SHA256:", case["response_sha256"])


## Expected pulse at one optical module

Bin contents are expected photoelectrons, not sampled detector hits. Time is relative to that module's recorded origin. Signed bin contents are preserved.


In [ ]:
trace = case["traces"][0]
edges = np.asarray(case["time_edges_ns"])
t = (edges[:-1] + edges[1:]) / 2
bins = np.asarray(trace["bins"])
fig, ax = plt.subplots(figsize=(9, 3.5))
for k, label in enumerate(["Direct", "Once scattered", "Multiple scattered"]):
    ax.step(t, bins[:, k], where="mid", label=label)
ax.step(t, bins.sum(axis=1), where="mid", color="black", label="Total")
ax.set(xlabel="Time relative to module origin (ns)", ylabel="Expected p.e. / bin", title=f"{case["title"]} — OM array index {trace["index"]}")
ax.legend()
fig.tight_layout()
plt.show()


## Photon Monte Carlo comparison

This is a separate recorded run. Ratios and uncertainty are read from its report. The historical acceptance rule applies to the total charge in a distance bin; it does not certify every scattering order or a global 3% precision.


In [ ]:
mc = next(x for x in data["matrix"] if x["id"] == case_id)
print("MC report:", mc["date"], mc["source"], "independent runs:", mc["runs"])
for row in mc["rows"]:
    total = row["total"]
    print(row["distance_m"], "LH/MC =", total["ratio"], "+/-", total["ratio_sigma"], "tested:", row["tested"], "passed:", row.get("passed"))


## Idea registry

All entries are imported. Registry classifications are source metadata; they are separate from measured outcomes.


In [ ]:
print("Ideas:", len(data["ideas"]), "stored case records:", data["record_count"])
for idea in data["ideas"]:
    print(idea["id"], "|", idea["status"], "|", len(idea["runs"]), "case records")
